# 18. Prepare the paired generalization study

Use a CPU runtime. This notebook fixes the experiment matrix, inspects saved progress,
and publishes the study configuration to R2. It trains no models and uploads no images.

The default study uses three seeds, three earlier learning orders, three forgotten tasks,
and incoming task 15. Each target occurs at positions 1, 4, and 8 across the orders.
Nine source models and nine shared learning-only controls support 27 paired comparisons.
There are 45 executable jobs: nine sources and 36 branches.

These are diagnostic settings from experiment 06. They are not an exact paper reproduction
and do not include a recovery probe. Notebooks 16 and 17 remain separate.
Source histories are trained fresh with one fixed configuration, including beta 0.1
for all eight lessons. They do not recreate the earlier mixed-setting checkpoint chain.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, make_plan, jobs, inspect_queue, run_queue, review
import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)


## 2. Choose the study

In [ ]:
STUDY_ID = "paired_generalization_v1"
BUCKET = "uncle-experiments"
MODE = "CHECK"  # CHECK reads only; PREPARE publishes the configuration.
if MODE not in ("CHECK", "PREPARE"):
    raise ValueError("Choose CHECK or PREPARE.")
if torch.cuda.is_available():
    raise RuntimeError("Switch to a CPU runtime for preparation.")
plan = make_plan(CODE_VERSION, REPO / "uncle/task_partition.json", STUDY_ID)
# Edit the matrix before PREPARE, using a new study ID for changed settings.
# For a smaller first study, keep order_01, seed 0, and forget task 3.
# Use STUDY_ID = "paired_generalization_pilot_v1" for that smaller matrix.
# plan["orders"] = {"order_01": plan["orders"]["order_01"]}
# plan["seeds"] = [0]
# plan["forget_tasks"] = ["3"]
# plan["keep_post_unlearning_checkpoint"] = True  # retain these for selected follow-up work
# plan["keep_branch_checkpoints"] = True  # retain all branch finals for later model inspection
from uncle.paired_study import validate_plan
validate_plan(plan)
print(json.dumps(plan, indent=2))


## 3. Inspect compute and storage requirements

In [ ]:
queue = jobs(plan)
from collections import Counter
counts = Counter(job["kind"] for job in queue)
paired_count = counts["unlearned"]
learning_lessons = sum(action == "learn" for job in queue for action, task in job["operations"])
approx_model_bytes = 538_000_000  # approximate observed model size, not a storage limit
retained_models = counts["source"]
if plan["keep_branch_checkpoints"]:
    retained_models += counts["control"] + counts["unlearned"]
if plan["keep_post_unlearning_checkpoint"]:
    retained_models += paired_count
print("Work units:", dict(counts))
print("Paired comparisons:", paired_count)
print("Learning lessons:", learning_lessons, "Unlearning requests:", paired_count)
print("Approximate final model storage:", round(retained_models * approx_model_bytes / 1e9, 2), "GB")
print("Two active resume slots add roughly 4.3 GB for a large job; actual size is measured when saved.")
print("Runtime is unknown until the first GPU sessions measure training, evaluation, and transfers.")
print("Resume saves include Adam and the original protection snapshot, so they exceed final model size.")
print("Sources remain available. Branch models are deleted after verified completion; reports and hashes remain.")
print("Temporary slots are removed only after the final model and report are verified.")
print("No existing experiment folders are deleted. No dataset images are uploaded.")


## 4. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab Secrets
or environment variables. Their values are never printed or saved in the notebook.
Use only one active writer per study.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value
store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)


## 5. Check or publish the configuration

In [ ]:
storage = StudyStore(store, plan, CACHE)
existing = storage.read(storage.prefix + "/study.json")
if existing is not None and existing != plan:
    raise ValueError("This study ID already has different settings. Restore them or choose a new ID.")
if MODE == "PREPARE":
    storage.prepare()
    print("Study configuration uploaded and verified. No training started.")
elif existing is None:
    print("New study. Inspect this matrix, then use MODE = PREPARE to save it.")
else:
    print("Saved study matches this configuration.")
rows = inspect_queue(plan, storage, progress=True)
print("Queue:", dict(Counter(row["status"] for row in rows)))
for row in rows:
    print(row["status"].ljust(10), row["id"], row["active"] or "")
print("Next: open notebook 19 with the same STUDY_ID in a GPU runtime.")
